# care-nudge training on Colab

Runtime → Change runtime type → **T4 GPU**. The cells run top to bottom: build the data, score two baselines, fine-tune with QLoRA, evaluate, then merge and publish the model the Space loads.

If the Colab GPU quota runs out, `train_kaggle.ipynb` has the same cells for Kaggle.

In [ ]:
# Cell 1: fresh copy of the code and the training stack
%cd /content
!rm -rf /content/care-nudge
!git clone https://github.com/SachiK0thari/care-nudge.git
%cd /content/care-nudge
!pip install -q -r requirements-dev.txt
# Kaggle/Colab preinstall an old torchao that this project never uses; PEFT raises on any torchao < 0.16
# when it loads an adapter onto a non-quantized model (evaluate.py, merge.py). Remove it.
!pip uninstall -y -q torchao

In [ ]:
# Cell 2: log in to Hugging Face (paste a write token)
from huggingface_hub import login
login()

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Cell 3: data + two baselines from the un-tuned base model: the same prompt, then the prompt plus worked examples.
!python scripts/build_dataset.py
# The two baselines take about 30 min each on a T4. Comment them out if reports/eval_base*.json already exist.
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --name base
!python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --few-shot --name base_fewshot

In [ ]:
# Cell 4: fine-tune, then evaluate the adapter on the test split and on the probe set
# && stops the chain if a step fails, so evaluate/merge never run against a missing adapter
!python scripts/train.py --push-to SachiK/carenudge-lora && \
  python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --name finetuned && \
  python scripts/evaluate.py --model Qwen/Qwen3-4B-Instruct-2507 --adapter outputs/carenudge-lora --test data/probes.jsonl --name finetuned_probes

In [ ]:
# Cell 5: merge the adapter and publish the model the Space will load
![ -f outputs/carenudge-lora/adapter_config.json ] && python scripts/merge.py --push-to SachiK/carenudge-qwen3-4b || echo "No adapter in outputs/carenudge-lora: Cell 4 training did not finish, skipping merge."